In [1]:
using CSV, DataFrames
using Statistics
using Random

In [2]:
# Load Training Data and Prepare Features for ORT Regression

# Read in the cleaned training dataset
df = CSV.read("../clean_data/train_data_features.csv", DataFrame)

# Extract the continuous target variable (Hub North price)
y = df.HB_NORTH

# Select all feature columns except:
#   - HB_NORTH (the target)
#   - interval_start_local (timestamp, not useful for tree splits)
feature_cols = Not([:HB_NORTH, :interval_start_local])
X_df = df[:, feature_cols]

# Replace any missing values with 0.0 because
# IAI trees do not accept missing entries
X_df = coalesce.(X_df, 0.0);

# === PREVIEW & DROP COLUMNS 120–121 AND 124–127 ===
cols_to_drop = vcat(120:121, 124:127)

println("Preview of columns to drop (indices $(collect(cols_to_drop))):")
println("Column names: ", names(X_df)[cols_to_drop])

# Show the first few rows of these columns (optional but useful)
display(first(X_df[:, cols_to_drop], 5))

# Drop them
X_df = X_df[:, Not(cols_to_drop)]
# ==================================================

# Convert feature table and target into standard Julia arrays
X = Matrix{Float64}(X_df);
y_vec = Vector{Float64}(y);

# X is the feature matrix for training an ORT
# y_vec is the continuous response vector


Preview of columns to drop (indices [120, 121, 124, 125, 126, 127]):
Column names: ["price_ramp_hist", "price_rolling_6h_hist", "price_lag1", "price_lag2", "price_lag4", "price_lag12"]


Row,price_ramp_hist,price_rolling_6h_hist,price_lag1,price_lag2,price_lag4,price_lag12
,Float64,Float64,Float64,Float64,Float64,Float64
1,-3.91,9.09167,-1.49,2.42,7.9,12.85
2,-1.11,4.22167,-2.6,-1.49,10.84,20.84
3,-2.86,1.935,-5.46,-2.6,2.42,40.05
4,-20.55,-3.71667,-26.01,-5.46,-1.49,24.07
5,-4.74,-10.6483,-30.75,-26.01,-2.6,45.81


In [3]:
size(X_df)

(17362, 122)

In [4]:
# Keep only the first 4000 rows
X_medium = X[1:4000, :];   
y_medium = y_vec[1:4000];      

In [ ]:
"""
Optimal Regression Tree (ORT)

Train an interpretable regression tree to predict HB_NORTH prices
from energy features using the IAI OptimalTreeRegressor.
"""

# Train an ORT with cross-validated hyperparameter tuning

function train_ort(X::Matrix{Float64}, y::Vector{Float64})
    # Basic dimension check
    n, p = size(X)
    @assert length(y) == n "X and y must have the same number of rows"

    # Train / validation split (for a final hold-out check)
    Random.seed!(15095)                  # make results reproducible

    idx = shuffle(1:n)                   # random permutation of indices
    train_frac = 0.8
    n_train = round(Int, train_frac * n)

    train_idx = idx[1:n_train]
    valid_idx = idx[n_train+1:end]

    X_train = X[train_idx, :]
    y_train = y[train_idx]

    X_valid = X[valid_idx, :]
    y_valid = y[valid_idx]

    # Grid search over ORT hyperparameters
    # - max_depth controls tree complexity (interpretability)
    # - minbucket controls minimum leaf size (regularization)
    grid_ort = IAI.GridSearch(
        IAI.OptimalTreeRegressor(
            random_seed = 15095,   # same seed for internal randomness
        ),
        max_depth = 2:6,           # try shallow to moderately deep trees
        minbucket = [50, 100, 200] # larger leaves = smoother predictions
    )

    # Cross-validated fitting
    # validation_criterion = :rmse because this is regression (I think this will happen by default)
    IAI.fit_cv!(grid_ort, X_train, y_train, n_folds = 3)

    # Extract the best-performing ORT from the grid search
    best_ort = IAI.get_learner(grid_ort)
    println("Best ORT hyperparameters: ", IAI.get_params(best_ort))

    # Evaluate on the hold-out validation set
    yhat_valid = IAI.predict(best_ort, X_valid)
    val_rmse = sqrt(mean((y_valid .- yhat_valid).^2))
    println("Validation RMSE (ORT): ", round(val_rmse, digits = 3))

    return best_ort
end

# Train the ORT on the medium feature matrix X and medium output y
best_ort = train_ort(X_medium, y_medium)


[ Warning: This copy of Interpretable AI software is for academic purposes only and not for commercial use.
┌ Warning: Interpretable AI license expires soon: 2025-12-31T00:00:00. If you need to renew, please send us the following machine ID:
└ 2f8e288356930cdc5c0eb6055219cb90358df6590b6e3c0f954024035c4234c9


In [ ]:
# Test set preparation for ORT (Optimal Regression Tree)

# 1. Read test CSV
df_t = CSV.read("../clean_data/test_data_features.csv", DataFrame)

# 2. Define target (y) and feature set (X)
y_t = df_t.HB_NORTH                # Vector{Float64}

# Drop the target and the timestamp from the features
feature_cols = Not([:HB_NORTH, :interval_start_local])
X_df_t = df_t[:, feature_cols]     # DataFrame of predictors

# 3. Handle missing values just like training
X_df_t = coalesce.(X_df_t, 0.0)

# === PREVIEW & DROP COLUMNS 120–121 AND 124–127 (MATCHING TRAINING SET) ===
cols_to_drop = vcat(120:121, 124:127)

println("Preview of test columns to drop (indices $(collect(cols_to_drop))):")
println("Column names: ", names(X_df_t)[cols_to_drop])

# Show first few rows of these columns
display(first(X_df_t[:, cols_to_drop], 5))

# Drop the columns to maintain feature alignment with training
X_df_t = X_df_t[:, Not(cols_to_drop)]
# =================================================================

# 4. Convert to matrix/vector for IAI
X_t      = Matrix{Float64}(X_df_t) # n × p matrix of features
y_vec_t  = Vector{Float64}(y_t)    # n-vector of targets

# Evaluate the ORT on the test set

rmse(y, yhat) = sqrt(mean((y .- yhat).^2))

yhat_ort_t = IAI.predict(best_ort, X_t)

test_rmse_ort = rmse(y_vec_t, yhat_ort_t)
println("Test RMSE (ORT) on HB_NORTH: ", round(test_rmse_ort, digits = 3))


In [ ]:
function baseline_day_before_test_rmse(y_test)
    H = 24
    
    # shift by 24 inside the test set itself
    y_pred = [fill(missing, H); y_test[1:end-H]]

    y_true = y_test[(H+1):end]
    y_pred_clean = y_pred[(H+1):end]

    return sqrt(mean((y_true .- y_pred_clean).^2))
end

rmse_baseline = baseline_day_before_test_rmse(y_vec_t)
println("Test-set baseline RMSE (day-before): ", rmse_baseline)

improvement = (rmse_baseline - test_rmse_ort) / rmse_baseline
println("Improvement over baseline: ", round(improvement*100, digits=2), "%")

In [ ]:
println("Test RMSE: ", test_rmse_ort)

In [ ]:
println("\nORT Variable Importance:")

imp = IAI.variable_importance(best_ort)   # returns a DataFrame with Feature + Importance

# Sort by Importance descending
sorted_imp = sort(imp, :Importance, rev=true)

# Pretty print
for row in eachrow(sorted_imp)
    println(row.Feature, ": ", round(row.Importance, digits=4))
end

In [ ]:
println(names(X_df)[[95, 121, 63, 57, 82, 62, 1, 97, 14, 21, 54, 49, 87, 44, 28, 45, 84, 35, 80, 89, 43, 8, 120, 81, 92, 78, 102, 60, 74, 4, 56, 47, 113, 103, 7]])